# Neural-symbolic fusion: neural ensemble + symbolic v4 (URFD)

Neural stream: the 3-seed neck-centred ST-GCN ensemble (`neural_ensemble.py`), out-of-fold on the test folds and each fold's own models on its validation videos. Symbolic stream: v4 (`symbolic_v4.py`), fitted on each fold's training videos only.

Fusions:
- **Paper (Eq. 11-13)**: (neural OR symbolic) AND symbolic support in the last 80 frames.
- **Gated veto**: neural alarm kept unless the neck is at 0.8 m or higher (no fitted parameter).
- **Veto + rescue**: gated veto, plus the symbolic FALL state on its own.
- **Weighted average**: w * p_neural + (1 - w) * p_symbolic, w chosen per fold on that fold's validation videos.
- **Stacking**: logistic regression on [logit p_neural, p_symbolic], fitted per fold on that fold's validation videos (the option the pre-registered rule selected).

All test numbers are post-hoc (the URFD test folds were seen before). Open from `notebooks/` or the project root and **Run All**.

In [1]:
import os, sys
import numpy as np
PROJECT_ROOT = os.environ.get("PROJECT_ROOT") or ("." if os.path.isdir("splits") else "..")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "fall_detection_models", "code", "stgcn_urfd"))
import symbolic_v4 as S4
import fusion_v4_experiments as X
X.ROOT = os.path.abspath(PROJECT_ROOT)
X.NPY = os.path.join(X.ROOT, "processed_data", "cfg_d31bfed741ab", "urfd_npy")
X.KIN = os.path.join(X.ROOT, "processed_data", "kinect_height")
REP = os.path.join(X.ROOT, "reports", "symbolic_v4")
split, data = X.load_all()
NV = X.neural_probs(os.path.join(REP, "neural_val.csv"), data)       # fold k models on fold k validation
NT = X.neural_probs(os.path.join(REP, "neural_test_oof.csv"), data)  # out-of-fold on test
neural = lambda part, v: NV[v] if part == "val" else NT[v]
SYM = {}
for f in split["folds"]:
    m = S4.fit([(data[v]["h"], data[v]["y"], data[v]["l3"]) for v in f["train"]])
    for part in ("val", "test"):
        for v in f[part]:
            SYM[(f["fold"], part, v)] = m.predict(data[v]["h"])
print("ready:", len(data), "videos")

ready: 70 videos


In [2]:
def run(method, k, part, v, w=None, lr=None):
    pn, s = neural(part, v), SYM[(k, part, v)]
    if method == "neural":   return (pn >= 0.5).astype(int)
    if method == "symbolic": return s["hard"]
    if method == "paper":    return X.fuse("F1", pn, s)
    if method == "veto":     return X.fuse("F4", pn, s)
    if method == "rescue":   return X.fuse("F5", pn, s)
    if method == "weighted": return X.fuse("F3", pn, s, w)
    if method == "stacking": return (X.lr_predict(lr, np.c_[X.logit(pn), s["p"]]) >= 0.5).astype(int)

# per-fold parameters from that fold's validation videos only
W, LR = {}, {}
for f in split["folds"]:
    k = f["fold"]
    def val_mcc(w):
        return X.metrics([(v, data[v]["y"], run("weighted", k, "val", v, w=w)) for v in f["val"]])["MCC"]
    W[k] = max((0.6, 0.7, 0.8, 0.9), key=val_mcc)
    Xv = np.vstack([np.c_[X.logit(NV[v]), SYM[(k, "val", v)]["p"]] for v in f["val"]])
    LR[k] = X.fit_lr(Xv, np.concatenate([data[v]["y"] for v in f["val"]]))
    print(f"fold {k}: weighted-average w = {W[k]}, stacking weights = {np.round(LR[k], 3)}")

fold 0: weighted-average w = 0.9, stacking weights = [ 2.237 -4.364  0.712]
fold 1: weighted-average w = 0.9, stacking weights = [ 1.479 -0.581  0.824]


## Results

In [3]:
names = {"neural": "Neural ensemble alone", "symbolic": "Symbolic v4 alone", "paper": "Fusion: paper Eq. 11-13",
         "veto": "Fusion: gated veto", "rescue": "Fusion: veto + rescue", "weighted": "Fusion: weighted average",
         "stacking": "Fusion: stacking (pre-registered choice)"}
FUSED = {}
print(f"{'':42s} {'VAL MCC':>8s} | {'TEST MCC':>8s} {'F1':>6s} {'Prec':>6s} {'Rec':>6s} {'Spec':>6s} {'Acc':>6s}  falls  ADL clean  alarms/min")
for meth, title in names.items():
    res = {}
    for part in ("val", "test"):
        rows = []
        for f in split["folds"]:
            k = f["fold"]
            for v in f[part]:
                p = run(meth, k, part, v, w=W[k], lr=LR[k]); rows.append((v, data[v]["y"], p))
                if part == "test": FUSED[(meth, v)] = p
        res[part] = X.metrics(rows)
    t = res["test"]
    print(f"{title:42s} {res['val']['MCC']:8.3f} | {t['MCC']:8.3f} {t['F1']:6.3f} {t['Precision']:6.3f} {t['Recall']:6.3f} "
          f"{t['Specificity']:6.3f} {t['Accuracy']:6.3f}  {t['falls_detected']}/{t['falls']}  {t['adl_clean']}/{t['adl']}  {t['fa_per_min']:.2f}")


                                            VAL MCC | TEST MCC     F1   Prec    Rec   Spec    Acc  falls  ADL clean  alarms/min
Neural ensemble alone                         0.678 |    0.547  0.619  0.570  0.678  0.909  0.874  28/30  23/40  4.43
Symbolic v4 alone                             0.316 |    0.498  0.577  0.482  0.718  0.863  0.841  30/30  0/40  20.33
Fusion: paper Eq. 11-13                       0.358 |    0.447  0.529  0.405  0.763  0.801  0.795  30/30  0/40  17.11
Fusion: gated veto                            0.660 |    0.635  0.683  0.743  0.632  0.961  0.911  28/30  23/40  7.85
Fusion: veto + rescue                         0.465 |    0.581  0.647  0.606  0.694  0.920  0.886  30/30  21/40  8.25
Fusion: weighted average                      0.645 |    0.545  0.617  0.568  0.677  0.908  0.873  28/30  23/40  5.03
Fusion: stacking (pre-registered choice)      0.891 |    0.489  0.571  0.512  0.647  0.890  0.853  28/30  23/40  5.44


In [4]:
# confusion matrices on the test folds
for meth in ("neural", "paper", "veto", "stacking"):
    rows = [(v, data[v]["y"], FUSED[(meth, v)]) for f in split["folds"] for v in f["test"]]
    m = X.metrics(rows)
    print(f"{names[meth]:42s} TN {m['tn']:5d}  FP {m['fp']:5d} | FN {m['fn']:5d}  TP {m['tp']:5d}")

Neural ensemble alone                      TN  9210  FP   923 | FN   581  TP  1222
Fusion: paper Eq. 11-13                    TN  8115  FP  2018 | FN   428  TP  1375
Fusion: gated veto                         TN  9739  FP   394 | FN   664  TP  1139
Fusion: stacking (pre-registered choice)   TN  9020  FP  1113 | FN   637  TP  1166


In [5]:
# save the fused test predictions (one array per video and method, frame order of *_frame_labels.npy)
np.savez(os.path.join(PROJECT_ROOT, "fusion_v4_test_predictions.npz"), **{f"{v}__{m}": p for (m, v), p in FUSED.items()})
print("saved fusion_v4_test_predictions.npz")

saved fusion_v4_test_predictions.npz
